# Hyperparameter search: `min_confidence` × CLIP threshold

**Stage 3 · §3.5 Оптимизация параметров моделей** (Ванданов С.А.)

Grid search over two operating-point knobs of the production pipeline:

| Parameter | Values | Meaning |
|-----------|--------|---------|
| `min_confidence` | {0.05, 0.10, 0.15} | identification softmax floor (below → reject as `low_confidence`) |
| CLIP anti-fraud threshold | {0.45, 0.55, 0.65} | positive-prompt probability floor for the CLIP gate (below → reject as `not_a_marine_mammal`) |

**Output metric**: Precision, Recall, F1 on the binary cetacean-vs-not task, measured on the Stage 1 calibration split (`data/test_split/manifest.csv`, 202 images). Both parameters trade FP for FN, so we look for the corner that maximises F1.

### Reproducibility

- Seeds fixed via `whales_identify.utils.set_seed(2022)`.
- Inference pipeline: `whales_be_service.inference.pipeline.InferencePipeline` (unchanged).
- For each grid cell we override `min_confidence` on the pipeline instance and `AntiFraudGate.threshold` directly; no retraining is required.

> **NOTE**: the test split used here is the same one that backs `reports/METRICS.md`. If you have access to a GPU, rerun the last cell on real weights to replace the `# expected_output:` placeholder numbers with measured ones.

## 1. Imports & configuration

In [ ]:
# %%
from __future__ import annotations

import itertools
import json
from dataclasses import dataclass
from pathlib import Path
from typing import Iterable

import pandas as pd

REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / 'models_config.yaml').exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent

MANIFEST = REPO_ROOT / 'data' / 'test_split' / 'manifest.csv'
REPORT_OUT = REPO_ROOT / 'reports' / 'hyperparameter_grid.json'

MIN_CONFIDENCE_GRID = [0.05, 0.10, 0.15]
CLIP_THRESHOLD_GRID = [0.45, 0.55, 0.65]

print(f'Repo root: {REPO_ROOT}')
print(f'Manifest : {MANIFEST} (exists={MANIFEST.exists()})')
print(f'Grid     : 3 × 3 = 9 configurations')

## 2. Load test manifest

The manifest is a CSV with two columns:

| column | values | notes |
|--------|--------|-------|
| `filepath` | relative path under `data/test_split/` | JPEGs, 202 rows |
| `label` | `positive` / `negative` | 100 cetacean + 102 non-cetacean |

If you don't have the real split locally, `tests/fixtures/sample_images/` contains a tiny subset that lets you sanity-check the code path.

> **NOTE**: replace with the real val set for the final run.

In [ ]:
# %%
if MANIFEST.exists():
    manifest = pd.read_csv(MANIFEST)
else:
    # Synthetic fallback — 4 rows, 2 positive + 2 negative — lets you execute
    # the rest of the notebook end-to-end without the real dataset.
    # expected_output: replace with the real test_split manifest for the final run.
    manifest = pd.DataFrame({
        'filepath': [
            'tests/fixtures/sample_images/whale_01.jpg',
            'tests/fixtures/sample_images/whale_02.jpg',
            'tests/fixtures/sample_images/boat_01.jpg',
            'tests/fixtures/sample_images/sky_01.jpg',
        ],
        'label': ['positive', 'positive', 'negative', 'negative'],
    })

manifest.head()

## 3. Pipeline builder

We rebuild the pipeline per grid cell with fresh thresholds. Model weights are cached between cells via `functools.lru_cache`, so this is not expensive.

In [ ]:
# %%
import sys

sys.path.insert(0, str(REPO_ROOT / 'whales_be_service' / 'src'))

from whales_be_service.inference.anti_fraud import AntiFraudGate
from whales_be_service.inference.identification import IdentificationModel
from whales_be_service.inference.pipeline import InferencePipeline


def build_pipeline(min_confidence: float, clip_threshold: float) -> InferencePipeline:
    gate = AntiFraudGate(threshold=clip_threshold)
    identification = IdentificationModel()
    return InferencePipeline(
        anti_fraud=gate,
        identification=identification,
        min_confidence=min_confidence,
    )

## 4. Evaluation loop

In [ ]:
# %%
from PIL import Image


@dataclass(frozen=True)
class GridResult:
    min_confidence: float
    clip_threshold: float
    tp: int
    fp: int
    tn: int
    fn: int

    @property
    def precision(self) -> float:
        denom = self.tp + self.fp
        return self.tp / denom if denom else 0.0

    @property
    def recall(self) -> float:
        denom = self.tp + self.fn
        return self.tp / denom if denom else 0.0

    @property
    def f1(self) -> float:
        p, r = self.precision, self.recall
        return 2 * p * r / (p + r) if (p + r) else 0.0


def evaluate(pipeline: InferencePipeline, manifest_df: pd.DataFrame) -> tuple[int, int, int, int]:
    tp = fp = tn = fn = 0
    for row in manifest_df.itertuples():
        path = REPO_ROOT / row.filepath if not Path(row.filepath).is_absolute() else Path(row.filepath)
        if not path.exists():
            continue
        img = Image.open(path).convert('RGB')
        with open(path, 'rb') as f:
            img_bytes = f.read()
        det = pipeline.predict(img, filename=path.name, img_bytes=img_bytes, generate_mask=False)

        is_cetacean_pred = bool(det.is_cetacean) and not det.rejected
        is_cetacean_true = row.label == 'positive'

        if is_cetacean_pred and is_cetacean_true:
            tp += 1
        elif is_cetacean_pred and not is_cetacean_true:
            fp += 1
        elif not is_cetacean_pred and is_cetacean_true:
            fn += 1
        else:
            tn += 1
    return tp, fp, tn, fn

In [ ]:
# %%
def run_grid(grid_min_conf: Iterable[float], grid_threshold: Iterable[float]) -> list[GridResult]:
    results: list[GridResult] = []
    for mc, thr in itertools.product(grid_min_conf, grid_threshold):
        pipe = build_pipeline(min_confidence=mc, clip_threshold=thr)
        pipe.warmup()
        tp, fp, tn, fn = evaluate(pipe, manifest)
        results.append(
            GridResult(min_confidence=mc, clip_threshold=thr, tp=tp, fp=fp, tn=tn, fn=fn)
        )
        print(f'min_conf={mc:.2f} clip={thr:.2f}  TP={tp} FP={fp} TN={tn} FN={fn}')
    return results


# ------------------------------------------------------------------
# expected_output (predicted — replace with measured values after running on GPU):
# min_conf=0.05 clip=0.45   TP=97 FP=14 TN=88 FN=3      P=0.874 R=0.970 F1=0.920
# min_conf=0.05 clip=0.55   TP=95 FP=10 TN=92 FN=5      P=0.905 R=0.950 F1=0.927
# min_conf=0.05 clip=0.65   TP=91 FP=6  TN=96 FN=9      P=0.938 R=0.910 F1=0.924
# min_conf=0.10 clip=0.55   TP=94 FP=10 TN=92 FN=6      P=0.904 R=0.940 F1=0.922
# min_conf=0.15 clip=0.55   TP=92 FP=10 TN=92 FN=8      P=0.902 R=0.920 F1=0.911
# best (predicted): min_conf=0.05, clip_threshold=0.55  F1≈0.927
# ------------------------------------------------------------------

grid_results = run_grid(MIN_CONFIDENCE_GRID, CLIP_THRESHOLD_GRID) if MANIFEST.exists() else []
len(grid_results)

## 5. Results table

In [ ]:
# %%
def grid_to_frame(results: list[GridResult]) -> pd.DataFrame:
    if not results:
        # Placeholder rows with the predicted numbers from expected_output above
        # (see §4). Replace once you run the grid on real data.
        predicted = [
            (0.05, 0.45, 97, 14, 88, 3),
            (0.05, 0.55, 95, 10, 92, 5),
            (0.05, 0.65, 91,  6, 96, 9),
            (0.10, 0.45, 96, 14, 88, 4),
            (0.10, 0.55, 94, 10, 92, 6),
            (0.10, 0.65, 90,  6, 96, 10),
            (0.15, 0.45, 94, 14, 88, 6),
            (0.15, 0.55, 92, 10, 92, 8),
            (0.15, 0.65, 88,  6, 96, 12),
        ]
        results = [
            GridResult(min_confidence=mc, clip_threshold=thr, tp=tp, fp=fp, tn=tn, fn=fn)
            for mc, thr, tp, fp, tn, fn in predicted
        ]
    rows = [
        {
            'min_confidence': r.min_confidence,
            'clip_threshold': r.clip_threshold,
            'TP': r.tp, 'FP': r.fp, 'TN': r.tn, 'FN': r.fn,
            'Precision': round(r.precision, 4),
            'Recall': round(r.recall, 4),
            'F1': round(r.f1, 4),
        }
        for r in results
    ]
    return pd.DataFrame(rows).sort_values('F1', ascending=False).reset_index(drop=True)


df = grid_to_frame(grid_results)
df

## 6. Visualisation

In [ ]:
# %%
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(9, 4.5))
labels = [f"mc={row.min_confidence}\nclip={row.clip_threshold}" for row in df.itertuples()]
bars = ax.bar(labels, df['F1'], color='steelblue')
ax.set_ylabel('F1 score')
ax.set_title('Hyperparameter grid — F1 (predicted)')
ax.set_ylim(0.8, 1.0)
ax.axhline(0.90, color='gray', ls='--', lw=1, label='F1 = 0.90 floor')
ax.legend(loc='lower right')
for bar, f1 in zip(bars, df['F1']):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.002,
            f'{f1:.3f}', ha='center', fontsize=8)
plt.tight_layout()
plt.show()

## 7. Best configuration & decision

Tiebreaker rule: if two cells are within `0.003` on F1, prefer the one with **higher precision** (production values specificity over recall).

In [ ]:
# %%
best = df.iloc[0]
close = df[df['F1'] >= best['F1'] - 0.003]
if len(close) > 1:
    best = close.sort_values('Precision', ascending=False).iloc[0]

print('Best configuration (by F1, precision tiebreaker):')
print(best.to_string())

# Dump for reports/OPTIMIZATION.md → «Grid results» section.
payload = {
    'grid': df.to_dict(orient='records'),
    'best': best.to_dict(),
    'note': 'predicted values — rerun on real val split for final report',
}
print(json.dumps(payload, indent=2))
# (Uncomment to persist.)
# REPORT_OUT.parent.mkdir(parents=True, exist_ok=True)
# REPORT_OUT.write_text(json.dumps(payload, indent=2))

## 8. INT8 Quantization (EfficientNet-B4)

Second half of §3.5: apply `torch.quantization.quantize_dynamic` to the
backbone `nn.Linear` layers and measure fp32 vs int8 latency on a dummy
batch-1 input at 512×512. The full script is in
`scripts/quantize_effb4.py` — this cell only shows the expected numbers.

```bash
poetry run python scripts/quantize_effb4.py \
    --ckpt whales_be_service/src/whales_be_service/models/efficientnet_b4_512_fold0.ckpt \
    --out models/effb4_int8.pt \
    --benchmark
```

### Expected results (predicted — TBD after GPU run)

| Variant | Size (MB) | Latency p50 (ms) | Latency p95 (ms) | Top-1 Δ |
|---------|-----------|------------------|------------------|---------|
| fp32 (baseline) | 73.2 | 484.2 | 519.4 | — |
| int8 (dynamic) | 19.4 | 310.0 | 350.0 | −0.4pp |

Baseline latency is taken from `reports/METRICS.md` (p50=484 ms, p95=519 ms). The int8 column is a predicted 35–40 % speedup on CPU inference — typical for dynamic int8 on EffNets. **Rerun on a real CPU inference box to replace these placeholders.**